### Data Cleaning for SQL

In [3]:
import pandas as pd

data_file_path = "./Data/company_list.csv"

company_list = pd.read_csv(data_file_path)

try:
    company_list["listed_share"] = company_list["listed_share"].str.replace(',', '', regex=True)
    company_list["listed_share"] = company_list["listed_share"].astype(float)
except Exception as e:
    print("Failed", e)
print(company_list["listed_share"].dtypes)


float64


### Data loading into the SQL

For `company_list`

In [4]:
import io
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

connection_string = os.getenv('CONNECTION_STRING')
conn = psycopg2.connect(connection_string)
cur = conn.cursor()



''' Making sure that align with the sql table '''
company_list['id'] = company_list.pop('id')

col_list = company_list.columns.tolist()
load_data = f'''
    COPY "company_list" ({', '.join(col_list)})
            FROM STDIN
    WITH (FORMAT CSV, HEADER TRUE);
'''


''' Not so important to reuse in our small case, focus in on SQL. '''
company_list['listed_share'] = company_list['listed_share'].astype(str).str.replace(',', '')
company_list['listed_share'] = company_list['listed_share'].replace({',': ''}, regex=True).astype(float)


company_list['total_paid_up_capital_rs'] = company_list['total_paid_up_capital_rs'].astype(str).str.replace(',', '')
company_list['total_paid_up_capital_rs'] = company_list['total_paid_up_capital_rs'].replace({',': ''}, regex=True).astype(float)


company_list['market_capitalization_rs'] = company_list['market_capitalization_rs'].replace({',': ''}, regex=True).astype(float)


company_list['ltp'] = company_list['ltp'].replace({',': ''}, regex=True).astype(float)

company_list['paid_up_rs'] = company_list['paid_up_rs'].replace({',': ''}, regex=True).astype(float)

print(company_list["listed_share"].dtype)

buffer = io.StringIO()
company_list.to_csv(buffer, index=False, header=False)
buffer.seek(0)

with open(data_file_path, 'r') as f:
    cur.copy_expert(load_data, buffer)

conn.commit()


float64


### Similar pattern to load another table.

In [5]:
import io
import os
import psycopg2
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

connection_string = os.getenv('CONNECTION_STRING')
conn = psycopg2.connect(connection_string)

cur = conn.cursor()


combined_price_history_file_path = "./Data/combined_price_history.csv"
combined_price_history = pd.read_csv(combined_price_history_file_path)

combined_price_history['id'] = combined_price_history.pop('id')

cols = combined_price_history.columns.tolist()
cols.insert(0, cols.pop())
combined_price_history = combined_price_history[cols] # re-arrange
conn.rollback() 

load_data = f"""
    COPY "combined_price_history" ({', '.join(cols)})
            FROM STDIN
    WITH (FORMAT CSV, HEADER TRUE);
"""

buffer = io.StringIO()
combined_price_history.to_csv(buffer, index=False, header=True)
buffer.seek(0)

with open(combined_price_history_file_path, 'r') as f:
    cur.copy_expert(sql=load_data, file=buffer)

print(combined_price_history.columns.tolist())
conn.commit()

['id', 'turnover', 'symbol', 'company_name', 'sector', 'symbol_link', 'serial_number', 'date_added', 'unlocked', 'high', 'low', 'ltp', 'percent_change', 'qty']
